In [0]:
# Databricks notebook: 06_top_transactions
# Purpose: Transform PhonePe top transaction JSON into a
# ranked state/district/pincode Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType,
    LongType,
    DoubleType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.top_transactions"

# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT TOP TRANSACTION FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

top_transaction_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "top") &
        (F.col("domain") == "transaction")
    )
    .alias("m")
)

top_transaction_source_df = (
    top_transaction_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    top_transaction_source_df
    .select("source_path")
    .distinct()
    .count()
)

print("\nSource validation")
print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"Expected {EXPECTED_SOURCE_FILES:,} files, "
    f"but found {actual_source_files:,}."
)

print("Top transaction source validation passed.")


# ============================================================
# 3. DEFINE EXPLICIT JSON SCHEMA
# ============================================================

top_transaction_metric_schema = StructType([
    StructField("type", StringType(), True),
    StructField("count", LongType(), True),
    StructField("amount", DoubleType(), True)
])

top_transaction_record_schema = StructType([
    StructField("entityName", StringType(), True),

    # Included for schema-drift tolerance.
    StructField("name", StringType(), True),

    StructField(
        "metric",
        top_transaction_metric_schema,
        True
    )
])

top_transaction_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "states",
                ArrayType(top_transaction_record_schema),
                True
            ),
            StructField(
                "districts",
                ArrayType(top_transaction_record_schema),
                True
            ),
            StructField(
                "pincodes",
                ArrayType(top_transaction_record_schema),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit top transaction schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_top_transaction_df = (
    top_transaction_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            top_transaction_schema
        )
    )
)

parse_summary = (
    parsed_top_transaction_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.states").isNull() &
                F.col("parsed_json.data.districts").isNull() &
                F.col("parsed_json.data.pincodes").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_top_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files          : {parse_summary['source_files']:,}")
print(
    f"Files without top data: "
    f"{parse_summary['files_without_top_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_top_data"] == 0, (
    "One or more source files do not contain any readable "
    "state, district, or pincode ranking data."
)

print("Top transaction JSON parsing passed.")


# ============================================================
# 5. FUNCTION TO EXTRACT A RANKING LEVEL
# ============================================================

def extract_transaction_ranking(
    source_df,
    array_name,
    geography_level
):
    return (
        source_df

        # posexplode preserves the original array order.
        # Array position 0 becomes rank 1.
        .select(
            "*",
            F.posexplode(
                F.col(f"parsed_json.data.{array_name}")
            ).alias("rank_index", "rank_record")
        )

        .select(
            F.col("file_scope"),

            F.lit(geography_level)
                .alias("geography_level"),

            F.when(
                F.col("file_scope") == "state",
                F.col("state_slug")
            )
            .otherwise(F.lit(None).cast("string"))
            .alias("parent_state_slug"),

            F.coalesce(
                F.trim(F.col("rank_record.entityName")),
                F.trim(F.col("rank_record.name"))
            ).alias("entity_name"),

            F.lower(
                F.regexp_replace(
                    F.coalesce(
                        F.trim(F.col("rank_record.entityName")),
                        F.trim(F.col("rank_record.name"))
                    ),
                    "[^a-zA-Z0-9]+",
                    "-"
                )
            ).alias("entity_slug"),

            (F.col("rank_index") + F.lit(1))
                .cast("int")
                .alias("rank_position"),

            F.col("year").cast("int").alias("year"),
            F.col("quarter").cast("int").alias("quarter"),
            F.col("period"),

            F.trim(
                F.col("rank_record.metric.type")
            ).alias("payment_instrument_type"),

            F.col("rank_record.metric.count")
                .cast("long")
                .alias("transaction_count"),

            F.col("rank_record.metric.amount")
                .cast("double")
                .alias("transaction_amount"),

            F.col("parsed_json.responseTimestamp")
                .alias("response_timestamp_epoch_ms"),

            F.col("relative_path"),
            F.col("source_path"),
            F.col("source_commit"),
            F.col("content_sha256"),
            F.col("ingested_at_utc")
        )
    )


# ============================================================
# 6. EXTRACT STATE, DISTRICT, AND PINCODE RANKINGS
# ============================================================

top_states_df = extract_transaction_ranking(
    parsed_top_transaction_df,
    "states",
    "state"
)

top_districts_df = extract_transaction_ranking(
    parsed_top_transaction_df,
    "districts",
    "district"
)

top_pincodes_df = extract_transaction_ranking(
    parsed_top_transaction_df,
    "pincodes",
    "pincode"
)

top_transactions_with_amount_df = (
    top_states_df
    .unionByName(top_districts_df)
    .unionByName(top_pincodes_df)
)

print("\nState, district, and pincode rankings extracted.")


# ============================================================
# 7. HANDLE OPTIONAL TRANSACTION AMOUNT
# ============================================================

populated_amount_rows = (
    top_transactions_with_amount_df
    .filter(F.col("transaction_amount").isNotNull())
    .count()
)

if populated_amount_rows == 0:
    top_transactions_final_df = (
        top_transactions_with_amount_df
        .drop("transaction_amount")
    )

    print(
        "transaction_amount removed because it is absent "
        "from every top transaction record."
    )
else:
    top_transactions_final_df = top_transactions_with_amount_df

    print(
        f"transaction_amount retained: "
        f"{populated_amount_rows:,} populated records."
    )


# ============================================================
# 8. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    top_transactions_final_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("geography_level").isNull() |
                F.col("entity_name").isNull() |
                F.col("rank_position").isNull() |
                F.col("payment_instrument_type").isNull() |
                F.col("transaction_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("transaction_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_transaction_counts"),

        F.sum(
            F.when(
                F.col("rank_position") <= 0,
                1
            ).otherwise(0)
        ).alias("invalid_rank_positions"),

        F.countDistinct("source_path")
            .alias("represented_source_files")
    )
    .first()
)

duplicate_rank_keys = (
    top_transactions_final_df
    .groupBy(
        "source_path",
        "geography_level",
        "rank_position"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if "transaction_amount" in top_transactions_final_df.columns:
    negative_amount_rows = (
        top_transactions_final_df
        .filter(F.col("transaction_amount") < 0)
        .count()
    )
else:
    negative_amount_rows = 0

print("\nData-quality results")
print(f"Total Silver rows           : {quality_result['total_rows']:,}")
print(
    f"Missing required values     : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative transaction counts : "
    f"{quality_result['negative_transaction_counts']:,}"
)
print(
    f"Negative transaction amounts: "
    f"{negative_amount_rows:,}"
)
print(
    f"Invalid ranking positions   : "
    f"{quality_result['invalid_rank_positions']:,}"
)
print(
    f"Source files represented    : "
    f"{quality_result['represented_source_files']:,}"
)
print(f"Duplicate ranking keys      : {duplicate_rank_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_transaction_counts"] == 0
assert negative_amount_rows == 0
assert quality_result["invalid_rank_positions"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_rank_keys == 0

print("Top transaction quality checks passed.")


# ============================================================
# 9. DISPLAY ROW COUNTS BY GEOGRAPHY LEVEL
# ============================================================

print("\nRows by geography level")

display(
    top_transactions_final_df
    .groupBy("geography_level")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count")
    )
    .orderBy("geography_level")
)


# ============================================================
# 10. SAVE MANAGED DELTA TABLE
# ============================================================

(
    top_transactions_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 11. VALIDATE SAVED TABLE
# ============================================================

saved_top_transactions_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_top_transactions_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final top transaction Delta-table validation passed.")


# ============================================================
# 12. DISPLAY SAMPLE
# ============================================================

preview_columns = [
    "geography_level",
    "parent_state_slug",
    "entity_name",
    "rank_position",
    "year",
    "quarter",
    "period",
    "payment_instrument_type",
    "transaction_count"
]

if "transaction_amount" in saved_top_transactions_df.columns:
    preview_columns.append("transaction_amount")

display(
    saved_top_transactions_df
    .select(*preview_columns)
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "rank_position"
    )
    .limit(30)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.top_transactions
Expected files: 1,258

Source validation
Expected source files: 1,258
Actual source files  : 1,258
Top transaction source validation passed.

Explicit top transaction schema created.

JSON parsing
Source files          : 1,258
Files without top data: 0
Top transaction JSON parsing passed.

State, district, and pincode rankings extracted.
transaction_amount removed because it is absent from every top transaction record.

Data-quality results
Total Silver rows           : 10,880
Missing required values     : 0
Negative transaction counts : 0
Negative transaction amounts: 0
Invalid ranking positions   : 0
Source files represented    : 1,258
Duplicate ranking keys      : 0
Top transaction quality checks passed.

Rows by geography level


geography_level,row_count,source_file_count
district,10540,1258
state,340,34



Silver Delta table created: workspace.silver.top_transactions

Saved-table validation
Saved rows              : 10,880
Source files represented: 1,258
First period            : 2018-Q1
Last period             : 2026-Q2
Final top transaction Delta-table validation passed.


geography_level,parent_state_slug,entity_name,rank_position,year,quarter,period,payment_instrument_type,transaction_count
district,null,bengaluru urban district,1,2018,1,2018-Q1,TOTAL,6616121
district,null,pune district,2,2018,1,2018-Q1,TOTAL,3440226
district,null,jaipur district,3,2018,1,2018-Q1,TOTAL,2740898
district,null,mumbai suburban district,4,2018,1,2018-Q1,TOTAL,1749592
district,null,rangareddy district,5,2018,1,2018-Q1,TOTAL,1746541
district,null,hyderabad district,6,2018,1,2018-Q1,TOTAL,1740752
district,null,medchal malkajgiri district,7,2018,1,2018-Q1,TOTAL,1643812
district,null,north twenty four parganas district,8,2018,1,2018-Q1,TOTAL,1467769
district,null,gurugram district,9,2018,1,2018-Q1,TOTAL,1427526
district,null,patna district,10,2018,1,2018-Q1,TOTAL,1406093


# Silver Top Transactions — Completion Summary

The `workspace.silver.top_transactions` Delta table was successfully created and validated.

## Results

- Total ranked records: **10,880**
- State-ranking records: **340**
- District-ranking records: **10,540**
- Source files represented: **1,258**
- Data coverage: **2018-Q1 to 2026-Q2**

## Table grain

Each record represents:

- One source geography
- One ranking geography level
- One ranking position
- One year and quarter

## Available data

- State and district rankings
- Ranking position
- Transaction count
- Payment instrument type
- Complete source lineage

## Source limitations

The pinned source version does not provide:

- Pincode ranking records
- Transaction amounts in top-transaction files

Transaction amounts remain available in `workspace.silver.map_transactions`.

## Analytical use

This table supports:

- Regional leaderboard analysis
- Rank-movement analysis
- Persistent top-performing region detection
- Emerging-market identification
- Ranking-stability and regional-dominance research